# Lab 07 · Observability, tracing, and swappable agents

**Prepared by Mustafa Assaf** · Lecture 3: Debate, trace, decide · 15–20 minute lab

## Make a council you can watch, trace, and change

**So far:** in Lab 06, the chair kept a transcript in a Python list. You could read it after the run, but you could not watch the council while it worked, follow a decision back to its evidence in one step, or add an architect without touching the chair's code.

**New:** three properties that a multi-agent system needs before anyone can trust it or change it. Each one comes from one small mechanism:

- **Observable:** every message passes through a *message bus* that writes it to the **logs**: a live log stream that you watch while the council runs, and a log file with one JSON record per message that anyone can read, search, and count afterwards. With a free LangSmith key, the same run also appears in **LangSmith** as a trace: a tree of steps with their inputs, outputs, and timings.
- **Traceable:** every message has an id and cites what it is based on: sources, or earlier messages. Following those links in the log file answers the question "who said what, based on what?", even after the program has stopped.
- **Swappable:** the council finds its agents in a *registry* and talks to each one through the same interface. Adding or replacing an agent takes one line, and the council's code does not change.

All three rest on a fourth property: **structured** messages. Every message is a typed record with fixed fields, so code can check it, log it, and follow its links. The case is Lab 06's Payments platform.

**Optional: see the run in LangSmith.** [LangSmith](https://smith.langchain.com/) is a hosted tracing tool for LLM applications. Create a free account and an API key, save the key in Colab Secrets as `LANGSMITH_API_KEY`, and run this in a new cell right after the setup cell, before Step 1:

```python
import os
from google.colab import userdata
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
```

Without it, nothing is sent anywhere, and the lab runs the same, on the logs alone.

**At a glance**

![Lab 07 at a glance: Minimalist (model agent) or Modernizer (model agent) or Rulebook (added in one line), then Schema check (typed Message), then Message bus (publish()), then Live log (time · level · line) or council.log (one JSON per line) or LangSmith (trace tree (optional)), then Chair (majority or escalate), then Evidence trail (trace() from council.log).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/07.svg)

## The idea in context

As soon as several agents work together, a new question appears: what is the system actually doing? With one model call you can read the reply. With a council, a decision comes out of many messages between many agents, and a wrong decision is hard to explain unless every step was recorded. Software engineering met the same problem with distributed systems, and answered it with observability: logs, metrics, and traces that let you see a running system from the outside.

Three properties carry over to agents. A system is observable when you can watch it work while it runs. It is traceable when every output can be followed back to who produced it and what it was based on, which is also called provenance. It is swappable when a part can be added or replaced through a fixed interface, without rewriting the rest. Logs are the first tool for all of this: a live stream to watch, and a log file of structured records to search, count, and trace afterwards. A bus that logs every message, and a registry of agents, give all three properties. This is the event-driven design that many large systems use: components publish events, and anything interested subscribes. Tracing tools such as LangSmith add the execution view: each run as a tree of steps, with inputs, outputs, and timings.

| Key term | Meaning |
| --- | --- |
| **Observability** | Seeing what a running system does from its outputs: logs, metrics, and traces. |
| **Trace** | A record of one run as a tree of steps, with each step's input, output, and timing; LangSmith shows traces of LLM applications. |
| **Log** | A time-stamped record of what happened; structured logs, such as JSON lines, can be searched and counted. |
| **Registry** | A lookup table of agents by name, so the system never hard-codes who takes part. |

## Pattern · Shared Memory

*Memory pattern from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide. Related: Registry & Adapter.*

| | |
| --- | --- |
| **Context** | Agents may work in separate contexts but still need a common source of truth for goals, progress, and handoffs. |
| **Problem** | Copying messages between agents duplicates context and creates inconsistent versions of the shared state. |
| **Solution** | Give the agents one shared memory, so that what one agent writes is immediately available to the others. |
| **Use when** | Agents share a project state; their own contexts should stay separate; a common task board is useful. |
| **Trade-off** | Reduces duplication, but concurrent updates need clear ownership to avoid overwrites and races. |
| **Flow** | Agent A → Shared memory → Agent B → Coordinated state |
| **Quality attributes** | Benefits: Interoperability, Reliability, Cost. Liabilities: Reliability, Security, Maintainability. |
| **In this lab** | The log is the shared memory: every message is written to it, and the live view and the tracer read it. Only the bus writes to the log, and a message cannot change once it is written, which addresses the pattern's main liability: concurrent writes cannot overwrite each other. The council's registry is the *Registry & Adapter* part. |

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`), and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit', 'langsmith']):
    %pip install -q "agentic-ai-coursekit[framework] @ https://github.com/mkassaf/agentic-ai-course/archive/9c78f2181b0f793c60b2b081c79ae74276026e7d.zip"
from coursekit import make_ask
MODE = "practice"
PROVIDER = "freeinference"  # or "gemini" or "deepseek"
ask = make_ask(mode=MODE, provider=PROVIDER)

## Step 1 · A typed message

`Message` is a *dataclass*: a class that only holds named fields. `frozen=True` makes a message read-only once it is created, so nobody can change what an agent said after the fact. Every message has an `id`, a `sender`, a `kind`, an `option`, a one-sentence `claim`, and `cites`: the ids of the sources, or of the earlier messages, it is based on. These links are what make the council traceable. `dataclasses.asdict` turns a message into a dictionary, ready to be written to the log as JSON. `traceable`, from LangSmith's SDK, is used in Step 4.

In [ ]:
import collections, dataclasses, itertools, json, logging, sys
from langsmith import traceable  # traces each run for LangSmith; does nothing without a key

SOURCES = {"R1": "Choose microservices only when the team has 8+ people and services must deploy independently.",
           "F1": "The Payments platform has 12 developers in 3 squads.", "F2": "Only one engineer runs operations."}

@dataclasses.dataclass(frozen=True)
class Message:  # the typed schema: every message on the bus has exactly these fields
    id: str
    sender: str
    kind: str          # "proposal", "rejected", "decision", or "escalation"
    option: str
    claim: str
    cites: tuple = ()  # ids of sources, or of earlier messages

## Step 2 · Logs: a live stream and a log file

Observability starts with logs. Python's `logging` module sends each *log record* to one or more *handlers*, and each handler formats and writes it somewhere. Our logger has two:

- `live` writes one readable line per message to the notebook, with the time and a *level*, as the council runs. This is the live view of the demo.
- `to_file` writes the same message to `council.log` as one JSON object per line, the format that log tools and dashboards search. `extra={"record": ...}` hands the message's JSON to that formatter.

`publish` is the bus: agents never print their own messages, they hand each one to `publish`, which logs it. A rejected message or an escalation is logged as a `WARNING`, so problems stand out in the live view and can be counted in the file. Nothing is printed yet.

In [ ]:
logger = logging.getLogger("council")
logger.setLevel(logging.INFO)
live = logging.StreamHandler(sys.stdout)  # the live view: one readable line per message, as it happens
live.setFormatter(logging.Formatter("%(asctime)s %(levelname)-7s %(message)s", "%H:%M:%S"))
to_file = logging.FileHandler("council.log", mode="w")  # the log file: one JSON object per line
to_file.setFormatter(logging.Formatter('{"time": "%(asctime)s", "level": "%(levelname)s", "message": %(record)s}'))
logger.handlers = [live, to_file]  # replaces the handlers of an earlier run of this cell
ids = itertools.count()  # message ids: m0, m1, ...

def publish(msg):  # the bus: every message is logged, as a warning when it needs attention
    level = logging.WARNING if msg.kind in ("rejected", "escalation") else logging.INFO
    text = f"{msg.id:<4}{msg.sender:<11}{msg.kind:<11}{msg.option:<18}cites {list(msg.cites)}"
    logger.log(level, text, extra={"record": json.dumps(dataclasses.asdict(msg))})
    return msg

## Step 3 · Agents behind one interface

For this council, an agent is any function that takes a case and returns a reply with an option, a claim, and the ids it cites. `model_architect` builds such a function around a model call, and passes the sources as JSON, so the model sees each id exactly: it returns the inner function `propose`, which remembers its stance and its practice reply. `council` is the *registry*: a dictionary from each agent's name to its function. The council's code finds its agents only there, so it never needs to know how an agent works inside.

In [ ]:
def model_architect(stance, example):  # an architect driven by a model; any function from case to reply can join
    def propose(case):
        return json.loads(ask(f"You are an architect on a council. {stance}\nSources: {json.dumps(SOURCES)}\n"
                              f"Case: {json.dumps(case)}\nReply with JSON only, no code fences: "
                              '{"option": "modular monolith" or "microservices", "claim": "one sentence", '
                              '"cites": ["source ids"]}', example=example))
    return propose

council = {  # the registry: the council's code finds its agents only here
    "Minimalist": model_architect("Prefer the simplest design that works.",
        '{"option": "modular monolith", "claim": "One operator cannot run many services.", "cites": ["F2"]}'),
    "Modernizer": model_architect("Prefer designs that let each squad release alone.",
        '{"option": "microservices", "claim": "Squads that release alone meet R1.", "cites": ["R1", "F1"]}')}
OPTIONS, CASE = ("modular monolith", "microservices"), {"team_size": 12, "independent_deploys": True}

## Step 4 · Run the council, and watch the logs and the trace

`run_council` asks every registered agent and checks each reply against the schema: a known option, and citations of real sources only. It publishes the reply as a `proposal`, or as a `rejected` warning. Then the chair publishes its own message: a `decision` if a strict majority of the grounded proposals agree, or otherwise an `escalation` warning for the human. The chair's message cites the proposals it is based on. The log lines appear as each message is published. The last line printed is the chair's message as the log file recorded it.

`@traceable` turns `run_council` into a *trace* in LangSmith, and `traceable(name=name)(propose)` wraps each agent's call, so every agent, whatever it is inside, becomes one step of that trace. The logs record *what was said*; the trace records *how the run went*: which step called which, with what input and output, and how long each took.

In live mode, the schema check matters: in our test runs, models cited `"[R1]"` with brackets, or `"Case"`, which is not a source. The logs showed those messages as `rejected` warnings, and they did not vote.

In [ ]:
@traceable(name="council")
def run_council(case):  # ask every registered agent, check each reply against the schema, then decide
    grounded = []
    for name, propose in council.items():
        r = traceable(name=name)(propose)(case)  # each agent is a step in the trace
        ok = r.get("option") in OPTIONS and bool(r.get("cites")) and set(r["cites"]) <= set(SOURCES)
        msg = publish(Message(f"m{next(ids)}", name, "proposal" if ok else "rejected", str(r.get("option")),
                              str(r.get("claim")), tuple(r.get("cites", ()))))
        grounded += [msg] if ok else []
    option, count = (collections.Counter(m.option for m in grounded).most_common(1) or [("none", 0)])[0]
    decided = 2 * count > len(grounded)  # a strict majority of the grounded proposals
    return publish(Message(f"m{next(ids)}", "Chair", "decision" if decided else "escalation",
                           option if decided else "no majority", f"{count} of {len(grounded)} grounded votes",
                           tuple(m.id for m in grounded if m.option == option or not decided)))

run_council(CASE)
print(open("council.log").readlines()[-1].strip())  # the same message, as the log file recorded it

## Step 5 · Trace a decision from the log, then add an agent

`trace` reads `council.log`, finds a message by its id, and follows its citations back until it reaches the sources. It needs nothing but the log file, so the same trace works after the program has stopped, or on another computer. Then a third agent joins: `rulebook` is Lab 00's rule, with no model at all. It has the same interface as the model architects, so adding it takes one line, and `run_council` runs unchanged with three agents.

In [ ]:
def trace(msg_id, depth=0):  # rebuilt from the log file alone: who said what, based on what
    msg = {r["message"]["id"]: r["message"] for r in map(json.loads, open("council.log"))}[msg_id]
    print("    " * depth + f"{msg['id']} {msg['sender']}: {msg['option']}. {msg['claim']}")
    for ref in msg["cites"]:
        if ref in SOURCES:
            print("    " * (depth + 1) + f"{ref}: {SOURCES[ref]}")
        else:
            trace(ref, depth + 1)

def rulebook(case):  # Lab 00's rule as an agent: no model at all, but the same interface
    both = case["team_size"] >= 8 and case["independent_deploys"]
    return {"option": "microservices" if both else "modular monolith", "claim": "Applies R1.", "cites": ["R1", "F1"]}

council["Rulebook"] = rulebook  # adding an agent is one line; run_council does not change
trace(run_council(CASE).id)

## What you should see

In Step 4, three log lines appear, each with a time and a level: the Minimalist proposes the modular monolith and cites F2, the Modernizer proposes microservices and cites R1 and F1, and the chair logs a `WARNING`: an escalation, because 1 vote of 2 is not a majority. Then the same escalation is printed as the JSON record from `council.log`, with its time, level, and message. Your times will differ. If you set a LangSmith key, open your LangSmith project: each run is a trace named `council`, with one step per agent.

In Step 5, four more log lines show the second run: three proposals, and a decision for microservices, 2 votes of 3. The trace, rebuilt from the log file, starts at the decision, `m6`, and shows the two proposals it cites, each with its sources. Between the two runs, `run_council` did not change: only the registry did.

## Try it

1. **A broken message.** Make the Minimalist's practice reply cite `"F9"`, and run Steps 3 and 4. The log shows a `rejected` warning, and the chair counts only grounded proposals.
2. **Count from the logs.** Read `council.log` with `[json.loads(line) for line in open("council.log")]`, and count the messages of each level and of each sender with `collections.Counter`. This is how a dashboard turns logs into metrics.
3. **Trace the dissent.** The trace of `m6` does not show the Minimalist's view. Change the chair's `cites` so that a decision also cites the dissenting proposals, as Lab 06 kept the dissent. Run Step 5 again.
4. **Quiet the live view.** Add `live.setLevel(logging.WARNING)` and run the council again. The live view now shows only the problems. Does the log file still record everything?
5. **Read the trace.** With a LangSmith key, open the second run's trace. Which agent took longest, and where do you see what each agent received and returned? Which question does the trace answer that the log does not?
6. **Swap a model.** Give `model_architect` a third parameter, `ask_model`, and use it instead of `ask`. Then build the Modernizer with `make_ask(mode=MODE, provider="gemini")`. Which lines of `run_council` changed?
7. **Use it in your project.** Your council must be observable live, traceable, and swappable. What will your logs record for each message, and which part of your design plays the bus and the registry of this lab?

## Takeaway

Three small mechanisms make a council something you can trust and change: a typed message that cites its basis, a bus that logs every message, and a registry behind one interface. The logs make the council observable, live and afterwards, and a tracing tool such as LangSmith shows each run as a tree of timed steps; the citations in them make each decision traceable; and the registry makes agents swappable. Lab 08 puts a human architect in charge of the council's decision.

## Read further

- [Python logging HOWTO](https://docs.python.org/3/howto/logging.html)
- [LangSmith](https://smith.langchain.com/) and its [tracing quickstart](https://docs.smith.langchain.com/)
- [OpenTelemetry: observability primer](https://opentelemetry.io/docs/concepts/observability-primer/)
- [Martin Fowler: What do you mean by "Event-Driven"?](https://martinfowler.com/articles/201701-event-driven.html)